In [4]:
!pip install bertopic Sastrawi pandas plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# 1. LOAD DATA
# Jangan lupa handling missing values. Teks kosong akan membuat BERTopic error.
df = pd.read_csv('dataset_berita_merge.csv')
docs = df['isi'].dropna().tolist()

# 2. PERSIAPAN STOPWORDS BAHASA INDONESIA
# Ini yang membedakan tugas asal-asalan dengan yang dikerjakan serius.
factory = StopWordRemoverFactory()
base_stopwords = factory.get_stop_words()

# Tambahkan custom stopwords yang saya temukan banyak tersebar di datasetmu!
# Jangan malas untuk mengecek kata apa lagi yang perlu dibuang nanti.
custom_stopwords = [
    'kompascom', 'baca', 'video', 'foto', 'nya', 'yg', 
    'dan', 'di', 'dari', 'ke', 'ini', 'itu'
]
stopword_list = base_stopwords + custom_stopwords

vectorizer_model = CountVectorizer(stop_words=stopword_list)

# 3. INISIASI BERTOPIC (MULTILINGUAL)
# Kita paksa model menggunakan embedding yang paham bahasa Indonesia
topic_model = BERTopic(
    language="multilingual", 
    vectorizer_model=vectorizer_model,
    verbose=True,
    calculate_probabilities=True # Set True agar visualisasi probabilitas bisa jalan seperti di Kaggle
)

# 4. TRAINING MODEL
# Ini akan memakan waktu. Biarkan jalan dan perhatikan log-nya.
topics, probs = topic_model.fit_transform(docs)

# 5. EVALUASI DAN VISUALISASI (Sama seperti yang kamu lihat di Kaggle)
# Lihat daftar topik
print(topic_model.get_topic_info())

# Tampilkan Bar Chart
fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
fig_barchart.show()

# Tampilkan Intertopic Distance Map (Visualisasi bulatan-bulatan klaster)
fig_intertopic = topic_model.visualize_topics()
fig_intertopic.show()

2026-04-26 11:51:34,999 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

2026-04-26 11:55:02,809 - BERTopic - Embedding - Completed ✓
2026-04-26 11:55:02,815 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-26 11:55:45,400 - BERTopic - Dimensionality - Completed ✓
2026-04-26 11:55:45,403 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-26 11:55:46,375 - BERTopic - Cluster - Completed ✓
2026-04-26 11:55:46,432 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 11:55:47,503 - BERTopic - Representation - Completed ✓


    Topic  Count                                Name  \
0      -1    576        -1_jadi_sebut_indonesia_lalu   
1       0    158      0_haji_jemaah_ibadah_berangkat   
2       1    150           1_sekolah_didik_guru_ajar   
3       2    133     2_kendara_mobil_listrik_baterai   
4       3    131       3_ekonomi_rupiah_purbaya_uang   
5       4     89           4_makan_tubuh_darah_sehat   
6       5     81                     5_ai_the_and_or   
7       6     54             6_poin_main_tim_tanding   
8       7     49              7_euro_jerman_tim_main   
9       8     45        8_digital_ai_usaha_indonesia   
10      9     44        9_kpk_partai_politik_korupsi   
11     10     37      10_campak_imunisasi_kasus_bayi   
12     11     37             11_sehat_bpjs_layan_pbi   
13     12     36        12_minyak_energi_imf_ekonomi   
14     13     32            13_iran_as_militer_rudal   
15     14     31  14_program_halal_masyarakat_bangun   
16     15     31   15_presiden_ketua_prabowo_ang

In [ ]:
# Asumsi: Kamu sudah merunning kode sebelumnya hingga baris ini:
# topics, probs = topic_model.fit_transform(docs)

# 1. Bar Chart (Topik dan Kata Kuncinya)
# Sama persis seperti di Kaggle. Menampilkan skor c-TF-IDF dari tiap kata di sebuah topik.
fig_barchart = topic_model.visualize_barchart(top_n_topics=8, n_words=10)
fig_barchart.show()

# 2. Intertopic Distance Map
# Visualisasi 2D interaktif (menggunakan UMAP) dari ukuran dan kedekatan antar topik.
fig_intertopic = topic_model.visualize_topics()
fig_intertopic.show()

# 3. Hierarchical Clustering
# Menunjukkan bagaimana topik-topik kecil bisa digabung menjadi topik besar (agglomerative clustering).
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.show()

# 4. Similarity Heatmap
# Menunjukkan matriks korelasi antar topik. 
# PERINGATAN: Ini akan error jika jumlah topik yang dihasilkan dari datamu terlalu sedikit (misal di bawah 3).
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap.show()

# 5. Document Projection (Peta Sebaran Dokumen Beritamu)
# Ini visualisasi paling berat. Memetakan 2.238 baris berita milikmu ke dalam ruang 2D.
# Jika RAM komputermu kecil atau tidak pakai Colab/GPU, eksekusi ini akan memakan waktu atau crash.
fig_documents = topic_model.visualize_documents(docs)
fig_documents.show()